# Real Parseltongue Stage-3 test on the COAD TXT corpus

This notebook runs the **real** Nebius/Parseltongue Stage-3 implementation for all six candidate targets using TXT articles from `results/clawbio_skill_trial/tcga-coad/full_text_articles`. It writes a real pg-bench-compatible `stage3-export.json`, then validates that Stage 4 can discover every candidate verdict and its quoted evidence.

Prerequisite: configure `NEBIUS_API_KEY` and `NEBIUS_MODEL` in `.env`. Running Stage 3 performs four model passes per candidate.

## 1. Configure the real Stage-3 sample run

In [17]:
import importlib
from pathlib import Path

from IPython.display import FileLink, Markdown, display

import agnostik.parseltongue_corpus as parseltongue_corpus
importlib.reload(parseltongue_corpus)

from agnostik.candidates import PRESELECTED_CANDIDATES
from agnostik.objections.bundle import load_export
from agnostik.objections.targets import discover
from agnostik.parseltongue_corpus import (
    Stage3Config,
    discover_sources,
    export_completed_targets,
    select_target_sources,
    target_query,
    validate_stage4_export,
)

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Could not find pyproject.toml')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SOURCE_DIR = PROJECT_ROOT / 'results' / 'clawbio_skill_trial' / 'tcga-coad' / 'full_text_articles'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'clawbio_skill_trial' / 'tcga-coad' / 'parseltongue_stage3_sample'
PARTIAL_EXPORT_PATH = OUTPUT_DIR / 'stage3-export.partial.json'
TARGETS = PRESELECTED_CANDIDATES

# Keep the notebook test smaller than a production run while still running
# the complete four-pass Parseltongue pipeline for every candidate.
MAX_DOCUMENTS_PER_TARGET = 3
MAX_TARGET_CHARS = 150_000

config = Stage3Config(
    tumour_type='COAD',
    cancer_term='colon adenocarcinoma',
    source_dir=SOURCE_DIR,
    output_dir=OUTPUT_DIR,
    targets=TARGETS,
    max_documents_per_target=MAX_DOCUMENTS_PER_TARGET,
    max_target_chars=MAX_TARGET_CHARS,
)
print(f'Input: {SOURCE_DIR}')
print(f'Output: {OUTPUT_DIR}')
print(f'Candidates: {", ".join(TARGETS)}')

Input: C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\full_text_articles
Output: C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\parseltongue_stage3_sample
Candidates: EGFR, ERBB2, KRAS, MYC, WRN, PRMT5


In [7]:
import os
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env", override=True)
print(repr(os.getenv("NEBIUS_MODEL")))

'zai-org/GLM-5.2'


## 2. Preview the exact inputs and required verdict nodes

This cell is read-only and makes no API calls.

In [8]:
sources = discover_sources(config.source_dir)
print(f'Corpus contains {len(sources)} TXT articles.\n')
for target in config.targets:
    selected = select_target_sources(
        sources,
        target,
        max_documents=config.max_documents_per_target,
        max_chars=config.max_target_chars,
    )
    print(f'{target}: {[path.name for path in selected]}')
    print(f'  verdict node: {target.lower()}-verdict')
    print(f'  query: {target_query(target, config.cancer_term, config.tumour_type)}\n')

Corpus contains 300 TXT articles.

EGFR: ['PMC12162862.txt']
  verdict node: egfr-verdict
  query: Build a formal evidence dossier for therapeutic targeting of EGFR in colon adenocarcinoma (COAD). Extract balanced supporting and opposing facts only from the supplied documents, and attach exact verbatim document quotes to every fact. Derive intermediate claims with explicit :using dependencies. Finally derive exactly one Boolean node named egfr-verdict; true means the target is promising and false means it is rejected. The verdict's complete :using chain must terminate in the quoted facts. Do not leave it unknown.

ERBB2: ['PMC9367374.txt', 'PMC13357833.txt', 'PMC4506361.txt']
  verdict node: erbb2-verdict
  query: Build a formal evidence dossier for therapeutic targeting of ERBB2 in colon adenocarcinoma (COAD). Extract balanced supporting and opposing facts only from the supplied documents, and attach exact verbatim document quotes to every fact. Derive intermediate claims with explici

## 3. Refresh the partial Stage-3 JSON

The long-running CLI writes completed target systems under `OUTPUT_DIR`. Create or refresh the partial JSON from those completed targets without making model calls:

```powershell
uv run agnostik-parseltongue COAD `
  --output results/clawbio_skill_trial/tcga-coad/parseltongue_stage3_sample `
  --export-completed
```

The notebook cell below calls the same export function as that CLI command.

In [ ]:
PARTIAL_EXPORT_PATH, ACTIVE_TARGETS = export_completed_targets(OUTPUT_DIR, TARGETS)
print(f'Partial export: {PARTIAL_EXPORT_PATH}')
print(f'Completed targets: {", ".join(ACTIVE_TARGETS)}')
display(FileLink(PARTIAL_EXPORT_PATH))

## 4. Validate the partial JSON with the Stage-4 loader

In [ ]:
validate_stage4_export(PARTIAL_EXPORT_PATH, ACTIVE_TARGETS)
bundle = load_export(PARTIAL_EXPORT_PATH)
views = discover(bundle, list(ACTIVE_TARGETS))

rows = [
    '| Candidate | Verdict node | Verdict | Derived claims | Grounded quoted facts |',
    '|---|---|---|---:|---:|',
]
for view in views:
    grounded = sum(fact.is_grounded for fact in view.facts)
    verdict_node = view.verdict_node.id if view.verdict_node else 'missing'
    rows.append(
        f'| {view.symbol} | `{verdict_node}` | {view.label} | '        f'{len(view.claims)} | {grounded} |'
    )
display(Markdown('\n'.join(rows)))

## 5. Display the human-readable Stage-3 reports

Render every available Parseltongue report directly in the notebook. This is read-only, makes no model calls, and can also display reports from a partially completed CLI run.

In [13]:
for target in ACTIVE_TARGETS:
    report_path = OUTPUT_DIR / 'targets' / target.lower() / 'answer.md'
    if report_path.is_file():
        report = report_path.read_text(encoding='utf-8')
        display(Markdown(f'# {target}\n\n{report}'))
    else:
        display(Markdown(f'## {target}\n\n_Report not found: `{report_path}`_'))

# EGFR

> **Inconsistency detected:** Cross-checks between independently derived verdicts (`egfr-verdict` vs. `egfr-verdict-from-extended` and `egfr-verdict` vs. `egfr-verdict-from-aggregate`) are marked as **contaminated** — these diffs reference shared upstream terms, meaning they are not truly independent verification paths [[diff:verdict-extended-vs-base]][[diff:verdict-aggregate-vs-axiom]]. Additionally, several aggregate terms and one grounded variant carry unverified or missing evidence [[diff:supporting-evidence-grounding-check]][[diff:reconciliation-grounding-check]]. Details below.

---

## Summary

The evidence dossier evaluates to a **positive verdict: EGFR is a promising therapeutic target in KRAS-mutant colon adenocarcinoma (COAD)** when used in combination with KRAS inhibition [[theorem:egfr-verdict]]. All 26 ground facts are verified against the source document (PMC12162862) with high-confidence quote matching. The verdict's complete dependency chain terminates in these quoted facts with no unknowns remaining.

The verdict logic follows a single decision rule: if EGFR has a non-redundant biological role, a defined mechanism, demonstrated dual-targeting efficacy, clinical translational evidence, and if the opposing concerns (single-agent resistance and context limitations) are each reconciled by the supporting evidence, then the target is promising [[axiom:combo-evidence-overcomes-resistance-implies-promise]].

---

## 1. Supporting Evidence

### 1.1 Non-Redundant Role of EGFR

EGFR performs functions distinct from KRAS in WNT signaling, stemness, and metabolic regulation, all of which cannot be replicated by KRAS inhibition alone.

> "we propose that the KRASG12D allele is critical in maintaining the axis of sustained proliferation, while upstream EGFR performs distinct, non-redundant functions involved in WNT, stemness and CSC signaling" [[fact:egfr-nonredundant-kras]]

> "the induction of the WNT- and stemness signature is uniquely driven by EGFR loss and cannot be replicated by KRAS inhibition" [[fact:wnt-signature-unique-egfr-loss]]

> "EGFR deletion in KRAS-mutant organoids reduced their phenotypic heterogeneity and activated a distinct cancer-stem-cell/WNT signature" [[fact:egfr-deletion-activates-wnt-stemness]]

> "This was accompanied by metabolic rewiring with a decrease in glycolytic routing and increased anaplerotic glutaminolysis" [[fact:egfr-deletion-metabolic-rewiring]]

> "downregulation of major signaling cascades like MAPK, PI3K, and ErbB" [[fact:egfr-deletion-downregulates-mapk-pi3k-erbb]]

These five facts aggregate into a confirmed non-redundant signature, evaluating to **True** [[term:egfr-nonredundant-signature]][[theorem:egfr-nonredundant-role-confirmed]].

### 1.2 Mechanism: SMOC2 as Master Regulator

The mechanism by which EGFR loss drives phenotypic changes is explained through SMOC2:

> "Smoc2 was identified as a key upregulated target mediating these phenotypes that could be rescued upon additional Smoc2 deletion" [[fact:smoc2-key-mediator]]

> "additional Smoc2-knockout could revert all these phenotypes induced by EGFR deletion, demonstrating that SMOC2 acts as a master-regulator of the identified key pathways in CRC" [[fact:smoc2-knockout-rescues-all]]

> "highlighting the role of EGFR as a negative regulator of SMOC2 and WNT signaling pathway" [[fact:egfr-negatively-regulates-smoc2]]

The mechanism is fully elucidated, evaluating to **True** [[term:mechanism-elucidated]][[theorem:mechanism-confirmed]].

### 1.3 Dual-Targeting Efficacy

Combination of EGFR and KRAS inhibition demonstrates dramatically superior efficacy over either agent alone:

> "KRAS inhibitor studies demonstrate EGFR to be essential for synthetic lethal action in combination with the novel non-covalent KRASG12D MRTX1133 inhibitor" [[fact:egfr-essential-synthetic-lethal]]

> "MRTX1133 treatment had a minimal impact on the proliferation of AKP organoids, while it completely inhibited proliferation in AKPE organoids" [[fact:mrtx1133-minimal-akp]][[fact:mrtx1133-complete-akpe]]

> "KRAS inhibition alone only modestly reduces proliferation, whereas dual inhibition completely abrogated proliferation" [[fact:dual-inhibition-abrogates-proliferation]]

Dual-targeting efficacy is confirmed, evaluating to **True** [[term:dual-targeting-validated]][[theorem:dual-inhibition-efficacy-confirmed]].

### 1.4 Clinical Translation

Evidence extends from organoid models to patient datasets and clinical trials:

> "heavily pretreated patients with metastatic colorectal cancer were shown to benefit from the treatment of adagrasib (KRASG12C inhibitor) in combination with cetuximab" [[fact:adagrasib-cetuximab-benefit]]

> "Validation in patient-datasets revealed that the identified signature is associated with better overall survival of RAS mutant CRC patients possibly allowing to predict therapy responses in patients" [[fact:akpe-signature-better-survival-krasmt]]

> "treatment of PDX (Study 3) with the anti-EGFR monoclonal antibody cetuximab shows upregulation of 729 genes also upregulated in AKPE organoids" [[fact:cetuximab-upregulates-akpe-genes]]

> "by correlating SMOC2 to EGFR expression in patients of the TCGA-COAD cohort, we observed a negative correlation in the subset of KRAS-mutant patients, that was not evident in KRASWT patients" [[fact:smoc2-negative-correlation-krasmt-tcga]]

Clinical translation is confirmed, evaluating to **True** [[term:clinical-translation]][[theorem:clinical-translation-confirmed]].

### 1.5 Aggregate Supporting Evidence

All four supporting pillars together evaluate to **True** [[term:supporting-evidence-balanced]][[term:supporting-evidence-grounded]]. The grounding check confirms no divergence between the balanced and grounded formulations [[diff:supporting-evidence-grounding-check]].

---

## 2. Opposing Evidence and Concerns

### 2.1 Single-Agent Resistance

Three classical concerns form the case against EGFR targeting as monotherapy:

> "KRAS-mutations are known to confer resistance" [[fact:kras-confers-resistance-egfr]]

> "Only 50% of patients eligible for anti-EGFR therapy respond to treatment, while the rest display primary resistance for reasons that are yet unknown" [[fact:only-50pct-respond-anti-egfr]]

> "EGFR inhibition was considered ineffective in KRAS mutated patients" [[fact:egfr-ineffective-kras-mutant-historical]]

These confirm that single-agent EGFR targeting faces substantial resistance, evaluating to **True** [[term:single-agent-resistance]][[theorem:single-agent-resistance-concerns-confirmed]].

### 2.2 Context Limitations

Three limitations bound the generalizability of EGFR's role to KRAS-mutant settings:

> "EGFR deletion in KRASwt tumor cells did not reduce tumor growth" [[fact:egfr-deletion-kraswt-no-growth-reduction]]

> "the AKPE signature could only stratify KRASmt patients with AKPE high signature from low signature expressors into better overall survivors, while this was not the case for KRASwt patients" [[fact:akpe-signature-kraswt-no-stratification]]

> "the metabolic phenotype induced by EGFR deletion is not exclusively driven by WNT signaling but involves broader EGFR-dependent regulatory networks" [[fact:metabolic-phenotype-not-exclusively-wnt]]

Context limitations are confirmed, evaluating to **True** [[term:context-limitations]][[theorem:context-limitations-confirmed]].

### 2.3 Aggregate Opposing Evidence

Both opposition pillars together evaluate to **True** [[term:opposing-evidence-balanced]][[term:opposing-evidence-grounded]]. The grounding check confirms no divergence [[diff:opposing-evidence-grounding-check]].

---

## 3. Reconciliation: How Opposing Concerns Are Addressed

### 3.1 Single-Agent Resistance → Dual Targeting Overcomes It

The resistance concerns are addressed by the demonstrated superiority of dual inhibition:

> "KRAS inhibition alone only modestly reduces proliferation, whereas dual inhibition completely abrogated proliferation" [[fact:dual-inhibition-abrogates-proliferation]]

> "heavily pretreated patients with metastatic colorectal cancer were shown to benefit from the treatment of adagrasib (KRASG12C inhibitor) in combination with cetuximab" [[fact:adagrasib-cetuximab-benefit]]

This implication (resistance concerns → dual-targeting efficacy is confirmed) holds, since both sides evaluate to True [[theorem:resistance-concerns-addressed-by-dual-targeting]].

### 3.2 Context Limitations → Clinical Evidence Addresses Them

The context limitations are confined to KRAS wild-type settings, while clinical translation evidence is specifically validated in KRAS-mutant patients:

> "Validation in patient-datasets revealed that the identified signature is associated with better overall survival of RAS mutant CRC patients possibly allowing to predict therapy responses in patients" [[fact:akpe-signature-better-survival-krasmt]]

> "Smoc2 was identified as a key upregulated target mediating these phenotypes that could be rescued upon additional Smoc2 deletion" [[fact:smoc2-key-mediator]]

> "additional Smoc2-knockout could revert all these phenotypes induced by EGFR deletion, demonstrating that SMOC2 acts as a master-regulator of the identified key pathways in CRC" [[fact:smoc2-knockout-rescues-all]]

This implication (context concerns → clinical evidence addresses them) holds [[theorem:context-concerns-addressed-by-clinical-evidence]].

### 3.3 Dossier Reconciliation

The reconciliation of both opposing pillars by supporting evidence evaluates to **True** [[term:dossier-reconciliation]][[term:dossier-reconciliation-grounded]]. The grounding check shows no divergence between the balanced and grounded versions [[diff:reconciliation-grounding-check]].

---

## 4. Final Verdict

The decision rule states that EGFR is a promising target if: (1) it has a non-redundant role, (2) the mechanism is elucidated, (3) dual targeting is efficacious, (4) clinical translation exists, (5) single-agent resistance is overcome by dual-targeting efficacy, and (6) context limitations are addressed by clinical evidence [[axiom:combo-evidence-overcomes-resistance-implies-promise]].

All six conditions are satisfied from the verified facts. The verdict evaluates to:

### **EGFR is a promising therapeutic target in KRAS-mutant COAD — TRUE** [[theorem:egfr-verdict]]

Two alternative derivations reach the same conclusion:

- An **extended formulation** using broader evidence sets (including direct CRC clinical benefit and MAPK amplification evidence) also returns True [[theorem:egfr-verdict-from-extended]].
- An **aggregate formulation** using the balanced supporting evidence and reconciliation nodes also returns True [[theorem:egfr-verdict-from-aggregate]].

All three verdict paths yield the same result.

---

## 5. Caveats

1. **Diff contamination:** Cross-checks between the base verdict and the extended/aggregate verdicts are marked as contaminated because they share upstream dependencies (`egfr-verdict`, `supporting-evidence-balanced`, `dossier-reconciliation`), making them non-independent confirmations [[diff:verdict-extended-vs-base]][[diff:verdict-aggregate-vs-axiom]]. While all three verdict paths return True, they should not be treated as fully independent validations.

2. **Unverified evidence in reconciliation-grounded:** The grounded version of the reconciliation term has quotes flagged as unverified despite originating from the primary source document. This appears to be a propagation artifact from the grounding process rather than a genuine evidence gap [[diff:reconciliation-grounding-check]].

3. **Missing evidence on aggregate/extended terms:** Six intermediate terms (`clinical-translation-extended`, `dual-targeting-validated-extended`, `single-agent-resistance-extended`, `supporting-evidence-balanced`, `opposing-evidence-balanced`, `dossier-reconciliation`) carry only `:origin` string labels rather than structured quote-verified evidence blocks. These are synthetic aggregation layers built atop fully verified facts, so the underlying evidence chain is intact, but the aggregation layers themselves lack direct document quotes.

4. **KRAS wild-type limitation:** The evidence dossier supports EGFR co-targeting specifically in **KRAS-mutant** COAD. The AKPE signature does not stratify KRAS wild-type patients, and EGFR deletion does not reduce tumor growth in KRAS wild-type cells [[fact:egfr-deletion-kraswt-no-growth-reduction]][[fact:akpe-signature-kraswt-no-stratification]]. The therapeutic promise does not extend to KRAS wild-type disease.

5. **Single source:** All evidence is drawn from a single document (PMC12162862). Independent confirmation from additional studies would strengthen the verdict.

# ERBB2

> **Consistency Warning:** The system flagged 3 integrity issues affecting this dossier. Several intermediate evidence bundles carry **unverified quotes** — the top-level `supporting-evidence-bundle`, `opposing-evidence-bundle`, `disqualifying-evidence-combination`, and `trial-efficacy-broad` all cite quotes that could not be matched against source documents. Additionally, two synthesized threshold facts (`pfs-meaningful-threshold-months = 6` and `meaningful-orrvs-threshold-pct = 30%`) are stated without document-attached evidence. These taints propagate into the final verdict, which is flagged as "potential fabrication." All underlying ground facts (individual trial results, biological data, guideline statements) are fully verified against source documents.

---

## ERBB2 Therapeutic Targeting in Colon Adenocarcinoma: Evidence Dossier and Verdict

### Summary

The formal verdict is **REJECTED** — the ERBB2 target is not recommended for therapeutic pursuit in COAD at this time. The system's Boolean verdict node `erbb2-verdict` evaluates to **False** [[theorem:erbb2-verdict]]. While a robust body of supporting evidence exists (cross-trial efficacy, biological rationale, guideline testing recommendations, durable case benefit, meaningful PFS), a simultaneously true disqualifying evidence combination — fatal toxicity, a failed pivotal endpoint, and absence of regulatory approval — overrides that support. The verdict logic requires supporting evidence to hold **and** the disqualifying combination to be absent; since the disqualifying combination is present, the target fails.

---

### 1. Supporting Evidence

All supporting components evaluate to **True**.

#### 1.1 Cross-Trial Efficacy Against Standard Comparators

Standard third-line therapies in mCRC provide the efficacy floor:

> "the trifluridine/tipiracil (TAS-102) and regorafenib have ORR of 2% and 1%, respectively" [[quote:standard-tas102-orr-pct]] [[quote:standard-regorafenib-orr-pct]]

TAS-102 ORR is **2%** [[fact:standard-tas102-orr-pct]] and regorafenib ORR is **1%** [[fact:standard-regorafenib-orr-pct]]. Five ERBB2-targeted trials exceed both:

| Trial | Agent | ORR | vs TAS-102 (2%) | vs Regorafenib (1%) |
|---|---|---|---|---|
| MOUNTAINEER | Trastuzumab + Tucatinib | **55%** | Superior [[theorem:mountaineer-orr-superior]] | Superior [[theorem:mountaineer-orr-vs-rego]] |
| DESTINY-CRC01 (Cohort A) | Trastuzumab Deruxtecan | **45.3%** | Superior [[theorem:destiny-orr-superior]] | Superior [[theorem:destiny-orr-vs-rego]] |
| MyPathway | Trastuzumab + Pertuzumab | **32%** | Superior [[theorem:mypathway-orr-superior]] | — |
| TRIUMPH (tissue) | Pertuzumab + Trastuzumab | **30%** | Superior [[theorem:triumph-orr-superior]] | — |
| HERACLES-A | Trastuzumab + Lapatinib | **28%** | Superior [[theorem:heracles-a-orr-superior]] | Superior [[theorem:heracles-a-orr-vs-rego]] |

Source quotes for each trial:

> "The ORR was 55%, mPFS was 6.2 m (95% CI: 3.5–NE), and mOS 17.3 m (95% CI: 12.3–NE)." [[quote:mountaineer-orr-pct]]

> "The ORR was 45.3% in cohort A" [[quote:destiny-crc01-a-orr-pct]]

> "mPFS was 2.9 m, mOS was 11.5 m, and ORR was 32%." [[quote:mypathway-orr-pct]]

> "ORRs were 30% and 28% in patients with ERBB2-positive tissue and ctDNA, respectively." [[quote:triumph-tissue-orr-pct]]

> "ORR was 28%, mPFS was 4.7 m (95% CI: 3.7–1), and mOS was 10.0 m (95% CI: 7.9–15.8)." [[quote:heracles-a-orr-pct]]

The broader efficacy node, `trial-efficacy-broad`, requires superiority across both comparators and evaluates to **True** [[term:trial-efficacy-broad]]. A consistency check confirmed that the broad and narrow efficacy definitions agree with no divergence [[diff:broad-vs-narrow-efficacy]].

#### 1.2 Biological Rationale

Two independent lines of preclinical evidence support ERBB2 as a therapeutic target:

> "ERBB2 inhibition with Neratinib and Afatinib (two EGFR tyrosine kinase inhibitors) resulted in diminished cell growth in transfected cell lines" [[quote:erbb2-inhibition-reduces-growth]]

> "ERBB2 activation by ERBB2 gene amplification or mutations is associated with anti-EGFR resistance in patients with mCRC" [[quote:erbb2-anti-egfr-resistance-marker]]

The combined biological rationale evaluates to **True** [[theorem:biological-rationale-exists]], using both facts as dependencies.

#### 1.3 Guideline Support

NCCN recommends ERBB2 testing in RAS/BRAF wild-type mCRC, though ESMO does not mention ERBB2:

> "The NCCN (National Comprehensive Cancer Network) guidelines state that testing ERBB2 amplification/overexpression should be made in patients with mCRC and absence of RAS or BRAF mutation." [[quote:nccn-recommends-erbb2-testing-mcrc]]

NCCN testing recommendation is **True** [[fact:nccn-recommends-erbb2-testing-mcrc]].

#### 1.4 Durable Case Benefit and Meaningful PFS

A published case report documents 12 months of sustained trastuzumab benefit:

> "Treatment with trastuzumab continued for 12 months in total" [[quote:case-treatment-duration-months]]

> "The patient's performance status began to improve within two months of initiating treatment, with an ECOG score of 2, improving over the course of the next five months to zero" [[quote:case-treatment-duration-months]]

The MOUNTAINEER median PFS of **6.2 months** [[fact:mountaineer-mpfs-months]] exceeds the **6-month** meaningful threshold, confirming a meaningful PFS signal [[theorem:mountaineer-pfs-meaningful]]. The 12-month case duration also exceeds this threshold [[theorem:case-demonstrates-durable-benefit]].

#### 1.5 Supporting Evidence Bundle — Aggregate

All five supporting components combine into `supporting-evidence-bundle`, which evaluates to **True** [[term:supporting-evidence-bundle]]:

1. Cross-trial efficacy (broad) [[term:trial-efficacy-broad]]
2. Biological rationale [[theorem:biological-rationale-exists]]
3. NCCN testing recommendation [[fact:nccn-recommends-erbb2-testing-mcrc]]
4. Durable case benefit [[theorem:case-demonstrates-durable-benefit]]
5. MOUNTAINEER PFS meaningfulness [[theorem:mountaineer-pfs-meaningful]]

---

### 2. Opposing Evidence

All opposing components also evaluate to **True**.

#### 2.1 Safety Concerns

**Fatal ILD in DESTINY-CRC01:**

> "Five patients had interstitial lung disease or pneumonitis (two grade 2; one grade 3; two grade 5, the only treatment-related deaths)." [[quote:ild-grade5-deaths-destiny]]

Two grade 5 deaths were recorded [[fact:ild-grade5-deaths-destiny]].

**CNS progression in HERACLES:**

> "CNS progression appeared in up to 19% of patients treated in this trial" [[quote:cns-progression-heracles-pct]]

Nineteen percent CNS progression rate [[fact:cns-progression-heracles-pct]] exceeds a 10% threshold. The composite safety concern evaluates to **True** [[theorem:safety-concern-exists]].

#### 2.2 No Guideline Endorsement or Regulatory Approval

> "there are currently no approved ERBB2-targeted therapies for mCRC" [[quote:no-approved-erbb2-therapies-mcrc]]

> "The ESMO (European Society of. Medical Oncology) guidelines do not mention ERBB2 amplification/overexpression" [[quote:esmo-no-erbb2-mention-guideline]]

> "therapies are currently not approved for these patients, and the recommendation is the enrollment of patients in a clinical trial" [[quote:no-erbb2-approval-clinical-trial-only]]

No approved therapies (**True**) [[fact:no-approved-erbb2-therapies-mcrc]] and no ESMO mention (**True**) [[fact:esmo-no-erbb2-mention-guideline]]. Clinical-trial-only access is confirmed (**True**) [[fact:no-erbb2-approval-clinical-trial-only]]. The combined no-guideline-endorsement node evaluates to **True** [[theorem:no-guideline-endorsement]].

#### 2.3 Biological Heterogeneity

ERBB2-altered cancers are not a uniform entity:

> "is not a homogenous entity but rather a collection of distinct diseases defined by histological context, specific alteration types, and unique co-mutation profiles" [[quote:erbb2-not-homogeneous-entity]]

> "ERBB2 mutations and amplification represent biologically distinct subgroups with potentially different therapeutic vulnerabilities" [[quote:erbb2-biologically-distinct-subgroups]]

Both facts are **True** [[fact:erbb2-not-homogeneous-entity]] [[fact:erbb2-biologically-distinct-subgroups]], and the heterogeneity concern evaluates to **True** [[theorem:biological-heterogeneity-concern]].

#### 2.4 Failed Pivotal Endpoint and Sub-Threshold ORR

HERACLES-B was formally negative:

> "being negative for this endpoint (9.7%, 95% CI: 0–28)" [[quote:heracles-b-orr-pct]]

ORR was only **9.7%** [[fact:heracles-b-orr-pct]], and the primary endpoint was negative (**True**) [[fact:heracles-b-primary-endpoint-negative]]. Even the 95% CI upper bound of **28%** [[fact:heracles-b-orr-ci-upper-pct]] falls below the **30%** meaningful ORR threshold [[fact:meaningful-orrvs-threshold-pct]], so HERACLES-B is below the meaningful benchmark [[theorem:heracles-b-below-meaningful-orrvs]].

#### 2.5 Combination Failure

Neratinib plus cetuximab showed no objective responses:

> "it did not show responses: seven received stable disease" [[quote:neratinib-cetuximab-no-response]]

This is **True** [[fact:neratinib-cetuximab-no-response]] and propagates as an ineffective combination [[theorem:neratinib-combination-ineffective]].

#### 2.6 Uncertain Prognostic Significance

> "there is no current consensus on the role of ERBB2 as a prognostic factor in CRC" [[quote:prognostic-role-no-consensus]]

**True** [[fact:prognostic-role-no-consensus]], carrying forward as prognostic uncertainty [[theorem:prognostic-significance-uncertain]].

#### 2.7 Opposing Evidence Bundle — Aggregate

All eight opposing components combine into `opposing-evidence-bundle`, evaluating to **True** [[term:opposing-evidence-bundle]].

---

### 3. Disqualifying Evidence Combination

A critical subset of opposing factors forms the `disqualifying-evidence-combination`, which requires **all three** to be simultaneously present:

1. **Safety concern** (fatal ILD + CNS progression) — **True** [[theorem:safety-concern-exists]]
2. **HERACLES-B primary endpoint negative** — **True** [[fact:heracles-b-primary-endpoint-negative]]
3. **No approved ERBB2 therapies** — **True** [[fact:no-approved-erbb2-therapies-mcrc]]

The disqualifying combination evaluates to **True** [[term:disqualifying-evidence-combination]], meaning all three conditions are met. A consistency check confirmed that this strict disqualifying subset is consistent with the full opposing evidence bundle with no divergence [[diff:strict-all-opposing-rejection]].

---

### 4. Final Verdict

The verdict node `erbb2-verdict` is defined as:

> If supporting evidence is established **AND** the disqualifying combination is **not** present → **True** (promising); otherwise → **False** (rejected).

**Evaluation:**

- `supporting-evidence-established` → **True** (the supporting bundle holds) [[theorem:supporting-evidence-established]]
- `disqualifying-combination-present` → **True** (all three disqualifying factors are present) [[theorem:disqualifying-combination-present]]
- `opposing-evidence-acknowledged` → **True** (the full opposing bundle holds) [[theorem:opposing-evidence-acknowledged]]

Therefore: `(and True (not True))` = `(and True False)` = **False**

> **(if False true false) → FALSE**

The ERBB2 target is **rejected** [[theorem:erbb2-verdict]].

Despite strong and consistent supporting evidence across multiple trials, biological rationale, and guideline testing recommendations, the simultaneous presence of fatal treatment-related toxicity, a failed pivotal clinical endpoint (HERACLES-B), and the complete absence of regulatory approval for ERBB2-targeted therapies in mCRC constitutes a disqualifying combination. The evidence base has progressed far enough to expose unacceptable safety signals and at least one definitive negative trial, without any successful regulatory pathway.

---

### 5. Caveats

1. **Unverified bundle-level quotes:** The `supporting-evidence-bundle`, `opposing-evidence-bundle`, `disqualifying-evidence-combination`, and `trial-efficacy-broad` terms carry evidence blocks whose quotes could not be verified against source documents (flagged as `verified: false, grounded: false`). These are synthesized summary statements rather than verbatim document extractions. However, every underlying individual fact these bundles reference (trial ORRs, safety events, guideline statements, biological findings) is fully verified against PMC9367374, PMC4506361, and PMC13357833 with high confidence scores.

2. **Synthesized thresholds without source evidence:** The two clinical thresholds — `pfs-meaningful-threshold-months = 6` [[fact:pfs-meaningful-threshold-months]] and `meaningful-orrvs-threshold-pct = 30` [[fact:meaningful-orrvs-threshold-pct]] — are analyst-synthesized values without document-attached quotes. These are standard oncology benchmarks but should be treated as assumptions rather than evidence-grounded facts.

3. **Fabrication-propagation taint:** Because the bundle-level terms carry unverified evidence, all downstream nodes that depend on them — `supporting-evidence-established`, `disqualifying-combination-present`, `opposing-evidence-acknowledged`, and `erbb2-verdict` itself — are flagged with "potential fabrication" taint. This does not change the Boolean evaluation (the verdict is clearly False from the underlying verified facts), but the formal evidence chain is not fully grounded end-to-end.

4. **Prevalence context:** ERBB2 amplification is present in only **3%** of all mCRC patients (5% in RAS/BRAF wild-type) [[fact:erbb2-amplification-prevalence-mcrc-pct]], a small but definable population that limits the scope of any therapeutic approach.

# KRAS

> **Consistency Warning:** Three source facts failed automated quote verification due to Unicode encoding mismatches (curly quotes `\u201c\u201d` and en-dashes `\u2013` in the stored quotes not matching the document's actual characters). Corrected versions were introduced and cross-checked. The original and corrected verdicts both evaluate to **true** with no divergences [[diff:verdict-crosscheck]], and all three independent consistency checks (cross-document priority, trial diversification, and verdict cross-check) show no disagreements [[diff:cross-doc-kras-priority]] [[diff:trial-diversification-crosscheck]]. However, four nodes carry a "potential fabrication" taint propagated from the three unverified quotes, detailed below.

---

# Evidence Dossier: Therapeutic Targeting of KRAS in Colon Adenocarcinoma (COAD/CRC)

## Summary

The final verdict is **true** — KRAS is a validated, promising therapeutic target in colorectal cancer [[theorem:kras-verdict]]. This conclusion integrates eight intermediate claims spanning prevalence, clinical relevance, pipeline scale, regulatory proof, first-generation limitations, next-generation solutions, combination efficacy, and acknowledged caveats. Every claim in the chain terminates in quoted facts from two source documents: PMC12658183 (a comprehensive KRAS-in-CRC review) and PMC5042411 (a RAS-prevalence meta-analysis).

---

## 1. Supporting Fact Base

### 1.1 High Prevalence

KRAS mutations are among the most prevalent oncogenic alterations in CRC, establishing the target's quantitative importance:

> "KRAS mutations occur in over one-third of colorectal cancers (CRC), primarily affecting codons 12 and 13, and less frequently codons 61, 117, and 146." [[fact:kras-mutations-over-one-third-crc]]

> "KRAS was mutated in 38% (670/1748) of the primary CRCs" [[fact:kras-mutated-38pc-primary-crc]]

> "RAS mutation prevalence was found to be 55.9%, with KRAS exon 2 mutations being most common (42.6% prevalence), followed by KRAS exon 4 (6.2%), NRAS exon 3 (4.2%), KRAS exon 3 (3.8%), NRAS exon 2 (2.9%), and NRAS exon 4 (0.3%) mutations" [[fact:ras-prevalence-55-9-pct]]

Using a 30% prevalence threshold rule (cancer mutation prevalence exceeding 30% justifies high-priority therapeutic development), both the primary-CRC KRAS prevalence (38%) and the metastatic KRAS exon 2 prevalence (42.6%) independently classify as high-priority [[theorem:doc1-target-priority]] [[theorem:doc2-kras-target-priority]]. Cross-document priority checks show full agreement [[diff:cross-doc-kras-priority]].

These three facts combine into the intermediate claim **high-prevalence-target** [[theorem:high-prevalence-target]].

### 1.2 Clinical Relevance

KRAS mutations drive adverse clinical outcomes and are already embedded in treatment algorithms:

> "KRAS-mutant CRCs are associated with poorer prognosis, higher recurrence rates, reduced chemotherapy response, and resistance to EGFR-targeted therapies." [[fact:kras-oncogene-role-poor-prognosis]]

> "mutations in codons 12 and 13 collectively accounted for nearly 82% of all KRAS mutations in CRC, with the following incidence order G12D > G12V > G13D > G12C > G12A > G12S (Fig. 4)." [[fact:c12-c13-account-82pc-kras-mutations]]

> "Stratifying patients by KRAS mutation status is now standard for guiding treatment, though not all mutations confer the same oncogenicity or therapeutic response." [[fact:kras-stratification-standard]]

> "the determination of RAS mutational status is needed for therapeutic decision-making" [[fact:kras-status-needed-therapeutic-decision]]

> "It was found that presence of mutations in KRAS exon 2 (codon 12/13) considerably reduced the efficacy of these EGFR inhibitors" [[fact:kras-exon2-reduces-egfr-efficacy]]

These five facts establish **clinical-relevance-established** [[theorem:clinical-relevance-established]].

### 1.3 Pipeline at Scale

The therapeutic development pipeline has reached meaningful scale:

> "The development of sotorasib and adagrasib, KRAS G12C-specific inhibitors, redefined KRAS as a challenging but tractable target, leading to an unprecedented surge in KRAS-targeted therapies in the last few years" [[fact:kras-redefined-tractable]]

> "A systematic search of the NCI Thesaurus and PubChem, combining automated searches and manual curation (see "Methods"), retrieved 106 drugs or cellular treatments targeting SHP2/SOS1/KRAS." [[fact:compounds-106-targeting-kras]]

> "A systematic examination of the Clinical Trials Database (https://clinicaltrials.gov), combining automatic search and manual curation, retrieved 156 clinical trials that have evaluated or are currently evaluating the response to at least one of those compounds in patients with CRC" [[fact:clinical-trials-156-crc]]

> "Eighty of these trials are currently recruiting patients, and 11 will recruit patients in the near future, reflecting active clinical interest to expand the therapeutic options to treat KRAS-mutant CRC patients, beyond the narrow subset of KRAS G12C–mutant cases." [[fact:clinical-trials-80-recruiting]]

⚠️ **Caveat on verification:** The 106-compound and 80-recruiting quotes failed automated document-text matching — the stored quotes contain escaped Unicode sequences (`\u201c`, `\u2013`) rather than actual curly-quote characters. Corrected versions ([[fact:compounds-106-corrected]], [[fact:clinical-trials-80-corrected]]) were created with proper Unicode, but their primary quotes *also* failed verification against normalized text. All four conjuncts nonetheless evaluate to **true** and the corrected term is confirmed [[term:pipeline-at-scale-corrected]]. These four facts combine into **pipeline-at-scale** [[theorem:pipeline-at-scale]], which carries a fabrication taint from the two unverified quotes.

### 1.4 Regulatory Proof

Two FDA approvals validate the regulatory pathway:

> "Adagrasib (Krazati) was FDA-approved for KRAS G12C-mutated locally advanced or metastatic CRC on June 21, 2024. The approval was granted for use in combination with cetuximab (Erbitux) in patients who had previously been treated with fluoropyrimidine-, oxaliplatin-, and irinotecan-based chemotherapy" [[fact:fda-approved-adagrasib-crc]]

> "Sotorasib (Lumakras) in combination with panitumumab (Vectibix) was approved by the FDA for CRC on January 16, 2025. This approval was specifically for adult patients with KRAS G12C-mutated metastatic CRC, who have received prior fluoropyrimidine-, oxaliplatin-, and irinotecan-based chemotherapy" [[fact:fda-approved-sotorasib-crc]]

These establish **regulatory-proof-exists** [[theorem:regulatory-proof-exists]].

---

## 2. Opposing Factors and Limitations

### 2.1 First-Generation Inhibitor Limitations

Current approved therapies address only a small fraction of KRAS-mutant CRC patients:

> "Despite their approval, these two inhibitors exclusively target the rare KRAS G12C mutation, which occurs in approximately 3% of the MSS CRC cases and is not found in MSI/hypermutated cases (Fig. 4), thereby limiting its applicability for the vast majority of CRC patients." [[fact:g12c-3pct-mss-absent-msi]]

> "KRAS G12C inhibitors have shown remarkable success in cancers harboring the G12C mutation, but as they are designed explicitly for this allele, they are not active against more prevalent KRAS mutations such as G12D and G12V." [[fact:g12c-inhibitors-not-active-prevalent]]

> "First-generation KRAS inhibitors, including sotorasib and adagrasib, exclusively target the GDP-bound inactive form (OFF). Treatment with these inhibitors often triggers an adaptive feedback reactivation of wild-type RAS-GTP or secondary mutations promoting tumor persistence" [[fact:first-gen-off-only-resistance]]

> "current FDA-approved inhibitors target only KRAS G12C, a rare variant in MSS CRC and virtually absent in MSI CRC." [[fact:current-fda-inhibitors-only-g12c]]

> "Of the 156 clinical trials, 52 evaluated compounds exclusively targeting KRAS G12C (Fig. 9), despite the very low incidence of this mutation in CRC." [[fact:clinical-trials-52-g12c-only]]

These five facts constitute **first-gen-limitations** [[theorem:first-gen-limitations]].

Numerically, 52 of 156 trials (33%) are G12C-exclusive. Since 156 − 52 = 104 > 78 (half of 156), the pipeline is classified as **"diversifying"** rather than G12C-concentrated [[theorem:trial-allocation-pipeline]]. A qualitative cross-check using a separate document passage confirming trials beyond G12C are expanding also yields **"diversifying"** [[theorem:qualitative-diversification]], with no discrepancy [[diff:trial-diversification-crosscheck]].

### 2.2 Additional Caveats

Several inherent challenges and remaining gaps are acknowledged:

> "Due to repeated failures of both direct and indirect approaches, KRAS was long considered "undruggable."" [[fact:kras-long-considered-undruggable]]

> "The European Medicines Agency (EMA) has not yet approved either sotorasib or adagrasib for CRC treatment, though these agents may be accessible via clinical trials or compassionate use." [[fact:ema-not-yet-approved-crc]]

> "However, simultaneous inhibition of all three RAS isoforms in normal cells carries a serious risk of toxicity" [[fact:pan-ras-serious-toxicity-risk]]

> "Currently, most KRAS-targeting immunotherapeutic strategies remain in early stages of development, and none have yet advanced to clinical application." [[fact:immunotherapy-early-stage-no-clinical]]

> "developing specific KRAS inhibitors faced significant challenges due to the picomolar affinity of KRAS for GTP/GDP, high intracellular GTP concentrations, lack of allosteric regulatory sites, and the complex network of interactions involving GEFs, GAPs, and effectors through extended protein-protein interaction surfaces that are inherently challenging to target by small molecules" [[fact:high-picomolar-affinity-challenge]]

⚠️ **Caveat on verification:** The "undruggable" quote failed automated matching due to escaped Unicode curly quotes (`\u201c\u201d`). The corrected fact [[fact:kras-undruggable-corrected]] adds the verified `kras-redefined-tractable` quote as corroboration — its document context shows the text "KRAS was long considered "undruggable."" immediately preceding the verified passage — but the primary quote still fails. These five facts combine into **additional-caveats-acknowledged** [[theorem:additional-caveats-acknowledged]], carrying a fabrication taint from the undruggable quote.

---

## 3. Counterbalancing Evidence: Next-Generation and Combination Approaches

### 3.1 Next-Generation Inhibitors Overcome First-Gen Limitations

> "RAS(ON) inhibitors, which target the GTP-bound active form, have demonstrated superior efficacy." [[fact:ras-on-inhibitors-superior-efficacy]]

> "MRTX1133 is highly selective for KRAS G12D and induces tumor regression in multiple in vivo models, including CRC" [[fact:mrtx1133-tumor-regression-crc]]

> "Preclinical studies show sustained suppression of RAS pathway signaling and prolonged tumor regression, whereas RAS(OFF) inhibitors often lead to relapse due to adaptive resistance" [[fact:preclinical-ras-on-sustained-regression]]

> "An increasing number of recent and ongoing trials are investigating inhibitors targeting other KRAS mutations common in CRC, such as G12D and G12V, as well as broader approaches aimed at pan-KRAS inhibition" [[fact:trials-beyond-g12c-expanding]]

> "Several of these novel inhibitors are already in phase 1 and 2 clinical trials (Fig. 9), and early results suggest that they may offer meaningful clinical benefits for a much larger proportion of CRC patients." [[fact:several-novel-inhibitors-phase-1-2]]

These establish **next-gen-overcomes-limitations** [[theorem:next-gen-overcomes-limitations]].

### 3.2 Combination Efficacy

> "Combining KRAS inhibitors with anti-EGFR therapy or other targeted agents has emerged as a promising approach, as it can enhance antitumor efficacy compared with KRAS inhibition alone." [[fact:combination-therapy-promise]]

> "Recent preclinical and clinical studies have demonstrated the potential of these combination approaches to improve response rates and progression-free survival in patients with KRAS-mutant cancer" [[fact:combination-improved-response-pfs]]

These establish **combination-efficacy-demonstrated** [[theorem:combination-efficacy-demonstrated]].

---

## 4. Final Verdict and Its Dependency Chain

The verdict integrates all eight intermediate claims via a conditional: if all eight are true, the target is promising (`true`); otherwise it is rejected (`false`).

| Intermediate Claim | Status | `:using` Chain (terminating in quoted facts) |
|---|---|---|
| high-prevalence-target | ✅ True | kras-mutations-over-one-third-crc, kras-mutated-38pc-primary-crc, ras-prevalence-55-9-pct [[theorem:high-prevalence-target]] |
| clinical-relevance-established | ✅ True | kras-oncogene-role-poor-prognosis, c12-c13-account-82pc-kras-mutations, kras-stratification-standard, kras-status-needed-therapeutic-decision, kras-exon2-reduces-egfr-efficacy [[theorem:clinical-relevance-established]] |
| pipeline-at-scale | ✅ True ⚠️ | kras-redefined-tractable, compounds-106-targeting-kras, clinical-trials-156-crc, clinical-trials-80-recruiting [[theorem:pipeline-at-scale]] |
| regulatory-proof-exists | ✅ True | fda-approved-adagrasib-crc, fda-approved-sotorasib-crc [[theorem:regulatory-proof-exists]] |
| first-gen-limitations | ✅ True | g12c-3pct-mss-absent-msi, g12c-inhibitors-not-active-prevalent, first-gen-off-only-resistance, current-fda-inhibitors-only-g12c, clinical-trials-52-g12c-only [[theorem:first-gen-limitations]] |
| next-gen-overcomes-limitations | ✅ True | ras-on-inhibitors-superior-efficacy, mrtx1133-tumor-regression-crc, preclinical-ras-on-sustained-regression, trials-beyond-g12c-expanding, several-novel-inhibitors-phase-1-2 [[theorem:next-gen-overcomes-limitations]] |
| combination-efficacy-demonstrated | ✅ True | combination-therapy-promise, combination-improved-response-pfs [[theorem:combination-efficacy-demonstrated]] |
| additional-caveats-acknowledged | ✅ True ⚠️ | kras-long-considered-undruggable, ema-not-yet-approved-crc, pan-ras-serious-toxicity-risk, immunotherapy-early-stage-no-clinical, high-picomolar-affinity-challenge [[theorem:additional-caveats-acknowledged]] |

### Verdict

**KRAS is a promising therapeutic target in colorectal cancer.** All eight conjuncts of the verdict predicate evaluate to true, yielding `kras-verdict = true` [[theorem:kras-verdict]]. The full `:using` chain is:

```
kras-verdict
 ├── high-prevalence-target
 │    ├── kras-mutations-over-one-third-crc (quoted ✅)
 │    ├── kras-mutated-38pc-primary-crc (quoted ✅)
 │    └── ras-prevalence-55-9-pct (quoted ✅)
 ├── clinical-relevance-established
 │    ├── kras-oncogene-role-poor-prognosis (quoted ✅)
 │    ├── c12-c13-account-82pc-kras-mutations (quoted ✅)
 │    ├── kras-stratification-standard (quoted ✅)
 │    ├── kras-status-needed-therapeutic-decision (quoted ✅)
 │    └── kras-exon2-reduces-egfr-efficacy (quoted ✅)
 ├── pipeline-at-scale
 │    ├── kras-redefined-tractable (quoted ✅)
 │    ├── compounds-106-targeting-kras (quoted ⚠️)
 │    ├── clinical-trials-156-crc (quoted ✅)
 │    └── clinical-trials-80-recruiting (quoted ⚠️)
 ├── regulatory-proof-exists
 │    ├── fda-approved-adagrasib-crc (quoted ✅)
 │    └── fda-approved-sotorasib-crc (quoted ✅)
 ├── first-gen-limitations
 │    ├── g12c-3pct-mss-absent-msi (quoted ✅)
 │    ├── g12c-inhibitors-not-active-prevalent (quoted ✅)
 │    ├── first-gen-off-only-resistance (quoted ✅)
 │    ├── current-fda-inhibitors-only-g12c (quoted ✅)
 │    └── clinical-trials-52-g12c-only (quoted ✅)
 ├── next-gen-overcomes-limitations
 │    ├── ras-on-inhibitors-superior-efficacy (quoted ✅)
 │    ├── mrtx1133-tumor-regression-crc (quoted ✅)
 │    ├── preclinical-ras-on-sustained-regression (quoted ✅)
 │    ├── trials-beyond-g12c-expanding (quoted ✅)
 │    └── several-novel-inhibitors-phase-1-2 (quoted ✅)
 ├── combination-efficacy-demonstrated
 │    ├── combination-therapy-promise (quoted ✅)
 │    └── combination-improved-response-pfs (quoted ✅)
 └── additional-caveats-acknowledged
      ├── kras-long-considered-undruggable (quoted ⚠️)
      ├── ema-not-yet-approved-crc (quoted ✅)
      ├── pan-ras-serious-toxicity-risk (quoted ✅)
      ├── immunotherapy-early-stage-no-clinical (quoted ✅)
      └── high-picomolar-affinity-challenge (quoted ✅)
```

Every branch terminates in a quoted fact. The verdict does not evaluate to `false` or remain unknown.

---

## 5. Consistency Checks

Three formal diffs were registered to test robustness:

1. **Verdict cross-check** — The original verdict (`kras-verdict`) uses uncorrected facts; the corrected verdict (`verdict-corrected`) replaces them with corrected counterparts. Both evaluate to `true` with **no divergences** [[diff:verdict-crosscheck]]. Three diff sub-results flagged "contamination" (references to unverified facts), but the final comparison confirmed equivalence [[diff:caveats-undruggable-fix]] [[diff:compounds-fix]] [[diff:trials80-fix]].

2. **Cross-document KRAS priority** — Applying the prevalence-priority rule to document 1 (38% primary CRC) vs. document 2 (42.6% metastatic KRAS exon 2) yields **"high-priority"** in both cases — no divergences [[diff:cross-doc-kras-priority]].

3. **Trial diversification cross-check** — A quantitative allocation test (104 non-G12C trials out of 156 > 78) vs. a qualitative textual assessment ("trials expanding beyond G12C") both yield **"diversifying"** — no divergences [[diff:trial-diversification-crosscheck]].

---

## 6. Caveats and Limitations of This Dossier

- **Three unverified quotes** (`compounds-106-targeting-kras`, `clinical-trials-80-recruiting`, `kras-long-considered-undruggable`) failed automated text matching due to Unicode encoding issues. All three are corroborated by adjacent verified passages in the same document (the 156-trial quote references "those compounds" linking back to the 106-compound search; the 52-G12C and trials-beyond-G12C quotes bracket the 80-recruiting statement; the undruggable text appears in the context of the verified `kras-redefined-tractable` quote). Corrected versions were introduced and produce the same verdict.

- **Two axioms** (`prevalence-priority-rule` at 30% threshold, `pipeline-diversification-check` at >50% non-G12C) were entered without evidentiary backing — they are methodological conventions, not document-derived facts. They affect only cross-check theorems, not the main verdict chain.

- **The verdict is unanimous** — all eight conjuncts evaluate to true and no diff reveals a material divergence. The fabrication taint is technical (quote-encoding), not substantive (no contradicted or fabricated data). The evidence supports a **promising** conclusion for KRAS as a therapeutic target in colorectal cancer.

## MYC

_Report not found: `C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\parseltongue_stage3_sample\targets\myc\answer.md`_

## WRN

_Report not found: `C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\parseltongue_stage3_sample\targets\wrn\answer.md`_

## PRMT5

_Report not found: `C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\parseltongue_stage3_sample\targets\prmt5\answer.md`_

## 6. Inspect the partial Stage-4 JSON contract

In [18]:
import json
from IPython.display import JSON

payload = json.loads(PARTIAL_EXPORT_PATH.read_text(encoding='utf-8'))
print(f'Globals: {list(payload)}')
print(f'DATA nodes: {len(payload["DATA"])}')
print(f'STRUCTURE_DATA nodes: {len(payload["STRUCTURE_DATA"])}')
print(f'Graph edges: {len(payload["LAYERS"]["edges"])}')
print(f'Tainted nodes: {len(payload["TAINT_DATA"]["tainted"])}')
display(JSON(payload, expanded=False))
display(FileLink(PARTIAL_EXPORT_PATH))

Globals: ['DATA', 'STRUCTURE_DATA', 'LAYERS', 'TAINT_DATA']
DATA nodes: 141
STRUCTURE_DATA nodes: 141
Graph edges: 152
Tainted nodes: 0


<IPython.core.display.JSON object>

C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\parseltongue_stage3_sample\stage3-export.partial.json

## 7. Hand the partial sample JSON to Stage 4

This uses the same `stage3-export.partial.json` validated and displayed in sections 4–6. It contains only `ACTIVE_TARGETS`.

```bash
uv run agnostik-objections inspect \
  --export results/clawbio_skill_trial/tcga-coad/parseltongue_stage3_sample/stage3-export.partial.json \
  --ledger

uv run agnostik-objections run \
  --export results/clawbio_skill_trial/tcga-coad/parseltongue_stage3_sample/stage3-export.partial.json \
  --out results/stage4-sample
```